# 🫁 Lung Cancer detection: Individual Model Pipeline

This notebook covers the complete analytical flow for identifying lung cancer types (**ACA, SCC**) and **Benign** tissue from histopathology slides.

## 1. Setup & Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from PIL import Image, ImageOps
import numpy as np
import matplotlib.pyplot as plt
import json
import os

print("Environment ready.")

## 2. Preprocessing & Normalization

We standardize histopathology slides by resizing and normalizing to ensure the AI focuses on cellular patterns rather than image intensity.

In [ ]:
def preprocess_lung_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img_resized = ImageOps.fit(img, (224, 224), Image.Resampling.LANCZOS)
    img_array = np.asarray(img_resized) / 255.0
    return img_resized, np.expand_dims(img_array, axis=0)

print("Lung preprocessing pipeline defined.")

## 3. Data Visualization

Visual check of the slide samples after standardization.

In [ ]:
# sample_path = 'images/lung_sample.jpg'
# if os.path.exists(sample_path):
#     img_view, _ = preprocess_lung_image(sample_path)
#     plt.imshow(img_view)
#     plt.title("Standardized Tissue Slide")
#     plt.axis('off')
#     plt.show()

## 4. Training History (12 Epochs)

The lung model utilized highly optimized features, reaching peak performance in **12 epochs**.

In [ ]:
history_file = '../json_files/lung cancer/training_history.json'
with open(history_file, 'r') as f:
    history = json.load(f)

epochs = range(1, len(history['accuracy']) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, history['accuracy'], 'green', label='Train Accuracy')
plt.plot(epochs, history['val_accuracy'], 'blue', linestyle='--', label='Val Accuracy')
plt.title('Lung Model: Accuracy Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history['loss'], 'green', label='Train Loss')
plt.plot(epochs, history['val_loss'], 'blue', linestyle='--', label='Val Loss')
plt.title('Lung Model: Loss Reduction')
plt.legend()

plt.show()

## 5. Model Prediction Engine

The model classifies samples into three distinct clinical categories.

In [ ]:
model_path = '../models/lung_cancer_model.h5'
lung_model = load_model(model_path)
class_labels = {0: "Adenocarcinoma (ACA)", 1: "Benign Texture", 2: "Squamous Cell (SCC)"}

def classify_lung_slide(image_path):
    _, model_input = preprocess_lung_image(image_path)
    prediction = lung_model.predict(model_input)
    
    class_idx = np.argmax(prediction, axis=1)[0]
    confidence = prediction[0][class_idx] * 100
    
    print(f"🔍 PREDICTION: {class_labels[class_idx]}")
    print(f"📊 CONFIDENCE: {confidence:.2f}%")

print("System ready for histopathology classification.")
lung_model.summary()